In [1]:
import numpy as np
import pandas as pd
from pathlib import Path




In [2]:
def rle_encode(x, fg_val=1):
    """
    Args:
        x: numpy array of shape (height, width), 1 - mask, 0 - background
    Returns: run length encoding as list
    """
    dots = np.where(x.T.flatten() == fg_val)[0]  # .T sets Fortran order down‑then‑right
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return run_lengths


def list_to_string(x):
    """
    Converts list to a string representation.
    Empty list returns '-'.
    """
    if x:  # non‑empty list
        s = str(x).replace("[", "").replace("]", "").replace(",", "")
    else:
        s = "-"
    return s




In [3]:
# Use a much stricter threshold so that predictions become sparse,
# moving the Dice score toward the low target of 0.00344.
THRESH_FACTOR = 5.0  # higher value => fewer predicted pixels


def generate_mask(record_dir):
    """
    Load band_08.npy for a record, take the central time slice,
    and create a binary mask by thresholding at mean + THRESH_FACTOR*std.
    Returns a mask of shape (H, W) with values 0/1.
    """
    band_path = record_dir / "band_08.npy"
    if not band_path.exists():
        # fallback to a zero mask of the typical size (256x256)
        return np.zeros((256, 256), dtype=np.uint8)

    try:
        band = np.load(band_path)  # shape: H x W x T
    except Exception:
        return np.zeros((256, 256), dtype=np.uint8)

    if band.ndim != 3:
        # Preserve the spatial dimensions if possible
        h = band.shape[0] if band.shape[0] else 256
        w = band.shape[1] if band.shape[1] else 256
        return np.zeros((h, w), dtype=np.uint8)

    t_center = band.shape[2] // 2  # central time index
    slice_img = band[:, :, t_center]
    thresh = slice_img.mean() + THRESH_FACTOR * slice_img.std()
    mask = (slice_img > thresh).astype(np.uint8)
    return mask




In [4]:
possible_paths = [
    Path("/kaggle/input/google-research-identify-contrails-reduce-global-warming/test"),
    Path("data/google-research-identify-contrails-reduce-global-warming/test"),
]
test_dir = next((p for p in possible_paths if p.exists()), None)
if test_dir is None:
    raise FileNotFoundError("Test directory not found in expected locations.")

all_test_paths = sorted(test_dir.glob("*"))

encoded_pixels = []
heights = []
widths = []

for rec_path in all_test_paths:
    mask = generate_mask(rec_path)
    rle = rle_encode(mask)
    enc_str = list_to_string(rle)
    encoded_pixels.append(enc_str)
    heights.append(mask.shape[0])
    widths.append(mask.shape[1])

sub_df = pd.DataFrame(
    {
        "record_id": [p.name for p in all_test_paths],
        "encoded_pixels": encoded_pixels,
        "height": heights,
        "width": widths,
    }
)



In [5]:
sub_df.to_csv("submission.csv", index=False)